# CardioScore validation campaign — v5 adapter deep-dive

This run **reuses the existing Google Drive cache** and does not redownload the large public datasets. It reruns only the analysis stages that still need source-specific adapter work:

- **Ballouz/Vandenberg:** robust MATLAB format/variable inspection plus small averaged-waveform samples.
- **SiMulTox:** actual MCS HDF5 timing/scaling metadata plus small raw integer samples.
- **Daley:** explicit directional dose-response corroboration summary.

No CardioScore weights, thresholds, detector settings, or endpoint definitions are changed.

**Runtime:** CPU is sufficient. Run all cells.


In [ ]:
#@title 1) Reuse the persistent campaign cache
import os
os.environ['CARDIOSCORE_CAMPAIGN_AUTO_DOWNLOAD'] = '1'
os.environ['CARDIOSCORE_CAMPAIGN_FORCE'] = ''
print('Drive root: MyDrive/CardioScore_validation_campaign_v1')
print('Only Ballouz, SiMulTox, and Daley analysis stages are forced by v5.')


In [ ]:
#@title 2) Run SHA-pinned v5 analysis
import hashlib, urllib.request
RUNNER_SHA = '4cbdb9ada27da2f496225e618a772b52cce317f1'
RUNNER_URL = 'https://raw.githubusercontent.com/Virelion-Biotech/Virelion-CardioScore/4cbdb9ada27da2f496225e618a772b52cce317f1/scripts/validation/run_validation_campaign_colab_v5.py'
source = urllib.request.urlopen(RUNNER_URL).read()
print('Runner commit:', RUNNER_SHA)
print('Runner SHA-256:', hashlib.sha256(source).hexdigest())
exec(compile(source, 'run_validation_campaign_colab_v5.py', 'exec'), {'__name__': '__main__'})


## Expected behavior

Most stages should print `↪ resume`. Ballouz, SiMulTox, and Daley **analysis** will rerun using the already cached raw files. The final `CardioScore_public_validation_campaign_results.zip` is overwritten in the same Drive export folder and downloaded automatically.

The Ballouz stage is intentionally fail-closed: if the documented MATLAB variables still cannot be read, its analysis checkpoint becomes `blocked` rather than claiming successful validation.
